<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# `dts devel` Build and Run

Project source on the base station needs to become an image that the Duckiedrone can run, and that image must reach the Duckiedrone's Docker daemon. `dts devel` connects those steps for Duckietown projects. We will choose where to build, where to deliver the image, and where to run it so each machine's role is clear.

## Use `dts devel` to build, deliver, and run a project

`dts devel` invokes Docker using the current Duckietown project's configuration. Run these examples from a valid project, following its deployment instructions for a Duckiedrone you manage. Building creates an image, delivery loads it into the destination daemon, and running creates a project container.

### Choose the build and delivery machines

Building on the base station keeps the build work off the Duckiedrone. Building directly on the Duckiedrone uses its processor architecture and leaves the image there. Separating these choices lets you use one machine to build and another to run.

`dts devel build` uses Docker Buildx to build the project's image. Its `-H` or `--machine` option selects the Docker socket or hostname where the image is built; the default is the local daemon. Its `-D` or `--destination` option selects where the completed image is delivered as part of the build step.

When the build machine and destination differ, `dts` saves the completed image from the build machine and loads it into the destination daemon.

Build the current project on the local base station:

```bash
dts devel build
```

Build the current project directly on the Duckiedrone:

```bash
dts devel build -H DUCKIEDRONE_NAME
```

Build locally for an `arm64v8` Duckiedrone, then deliver the image to it:

```bash
dts devel build -a arm64v8 -D DUCKIEDRONE_NAME
```

Build on another Docker host, then deliver to the Duckiedrone:

```bash
dts devel build -H BUILD_HOST -a arm64v8 -D DUCKIEDRONE_NAME
```

In the final example, `BUILD_HOST` is the hostname or Docker socket used for the build. The image architecture must match what the destination can run. A build on an arm64v8 Duckiedrone can detect that target architecture automatically; a build on an amd64 base station normally needs `-a arm64v8` before delivery to an arm64v8 Duckiedrone.

A Docker context stores connection configuration for the Docker client. The `dts devel` host argument takes a hostname or Docker socket directly for one command. Use the Duckiedrone hostname with `-H`, keeping context names for `docker --context` commands.

When given a hostname, `dts` resolves it and uses the configured TCP Docker endpoint, which defaults to port `2375`. This differs from the SSH-backed context in [Notebook 12](./12-remote-duckiedrone-docker-contexts.ipynb). Use the connection supplied by the Duckiedrone's setup and project instructions: an unencrypted Docker endpoint exposes daemon operations to machines that can reach it.

### Choose the run machine

`dts devel run` creates and runs a container from the current project's image. Its `-H` or `--machine` option selects the Docker socket or hostname where that container runs.

Run the project image locally with the normal local project mount:

```bash
dts devel run
```

Run the project image on the Duckiedrone:

```bash
dts devel run -H DUCKIEDRONE_NAME
```

A local run can mount the current project directly into the container. For a normal remote run with mounting enabled, `dts` first synchronizes the project to the remote machine because a bind mount uses files on the daemon's host. It then runs the container there. `--no-mount` disables both the project mount and that automatic synchronization. The `--sync` option is valid only with a remote `-H` target.

The build, delivery, and run choices determine where the image and container appear. Use the host-specific inventories from [Notebook 12](./12-remote-duckiedrone-docker-contexts.ipynb) when checking those results.

## Further reading

See the `dts devel build --help` and `dts devel run --help` output for capabilities available in the version of `dts` installed on your base station. The [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) covers physical Duckiedrone management.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.

---

# `dts code` Workbenches

An LX project needs its dependencies, source, and startup process to work together each time you test a change. A __workbench__ is a container configured to run that project with the libraries and tools it needs. `dts code` uses the project's build settings and startup script to create it. We will build its environment, start the project's process, and open a second shell for inspection while that process runs.

## Use `dts code` for project workbenches

Run these commands from a project that supplies the required build settings and startup script, following its setup instructions. Building and starting a workbench create project resources on the selected machine.

The project's __recipe__ provides its build instructions and dependencies; a __launcher__ is the script that starts its process. `dts code build -R DUCKIEDRONE_NAME` prepares the recipe and delegates the image build to `dts devel build`. In the usual form, `-R` names the Duckiedrone and becomes the build machine unless the command is run with `--local`. Build from the project root so the workbench has the project's required dependencies:

```bash
dts code build -R DUCKIEDRONE_NAME
```

Start the workbench after the build. `dts code workbench` prepares the same recipe, invokes `dts devel run` for the Duckiedrone, and starts the project's default launcher in a container:

```bash
dts code workbench -R DUCKIEDRONE_NAME
```

Leave that terminal running. From a second terminal, open a shell in the existing workbench to inspect files or run tools in the same configured environment:

```bash
dts code workbench -R DUCKIEDRONE_NAME --shell
```

`--shell` opens a shell in the running container, so the first workbench command must already be running. It gives the second terminal the same dependencies and project configuration as the launcher. After changing code, follow the project's instructions to rebuild and restart the workbench as needed.

Use the project's supplied `-R DUCKIEDRONE_NAME` form. For `dts code workbench`, `-R` names the connected Duckiedrone and normally determines where the workbench runs. An explicit `-H` overrides the Docker machine when a project needs it to differ from the Duckiedrone; `--local` requests a local build or workbench. These options select targets for `dts`; Docker contexts are configured separately.

## Further reading

See the output from `dts code build --help` and `dts code workbench --help` for capabilities available in the version of `dts` installed on your base station.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.

---

# Virtual Duckiedrone Connections

A virtual Duckiedrone runs its services on the base station, but its internal Docker inventory is separate from the base station's practice resources. Choosing the right shell helps you find those services. We will locate the virtual Duckiedrone's container, enter it, and connect each inspection task to the appropriate path.

## Connect to a virtual Duckiedrone

A virtual Duckiedrone runs as a local base-station container. `dts duckiebot virtual connect DUCKIEDRONE_NAME` finds the container named `dts-virtual-DUCKIEDRONE_NAME` and opens an interactive shell with `docker exec`. The connection enters that local container.

Use [Notebook 11](./11-docker-contexts-and-local-targets.ipynb) to identify the base station's local Docker context and check for remote overrides. Replace `LOCAL_DOCKER_CONTEXT` with that context name. The filtered listing shows whether the intended container exists and its current state:

```bash
docker --context LOCAL_DOCKER_CONTEXT container ls --all \
  --filter 'name=^/dts-virtual-DUCKIEDRONE_NAME$'
```

If it is running, open its interactive shell:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

Inside that shell, `docker image ls` and `docker container ls` inspect the virtual Duckiedrone's own Docker daemon and services. This is analogous to running Docker commands in a physical Duckiedrone's SSH shell.

To diagnose a connection error, use `dts duckiebot virtual list` to check its state. For a Duckiedrone listed as `Down`, start it with `dts duckiebot virtual start DUCKIEDRONE_NAME` and reconnect. If it is absent, follow the virtual Duckiedrone setup procedure to create it. The listed state helps you choose between starting an existing Duckiedrone and completing its setup.

## Choose the right connection path

[Table 1](#table-1) connects each goal to the shell or context that reaches its resources.

<table id="table-1" class="lx-table">
  <caption>Table 1: Connection paths for Docker and Duckiedrone goals.</caption>
  <thead>
    <tr>
      <th>Goal</th>
      <th>Connection path</th>
      <th>Docker target</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Complete the local Docker exercises</td>
      <td>Local Docker context</td>
      <td>Learner-owned <code>lx-docker-*</code> resources on the base station</td>
    </tr>
    <tr>
      <td>Inspect a physical Duckiedrone through its own shell</td>
      <td>SSH, then a read-only Docker command</td>
      <td>Docker daemon on the physical Duckiedrone</td>
    </tr>
    <tr>
      <td>Inspect a physical Duckiedrone from the base station</td>
      <td>Explicit Duckiedrone Docker context</td>
      <td>Docker daemon on the physical Duckiedrone</td>
    </tr>
    <tr>
      <td>Open a shell in a running virtual Duckiedrone</td>
      <td><code>dts duckiebot virtual connect DUCKIEDRONE_NAME</code></td>
      <td>Local <code>dts-virtual-DUCKIEDRONE_NAME</code> container</td>
    </tr>
  </tbody>
</table>

The connection path determines which inventory a Docker command inspects. Use the base-station daemon for the `lx-docker-*` exercises, and the Duckiedrone's daemon to observe its services. This keeps practice cleanup separate from the services supporting the virtual or physical Duckiedrone.

## Further reading

See Docker's official guide to [contexts](https://docs.docker.com/engine/manage-resources/contexts/) and the [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) for physical Duckiedrone management.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.

---

# Virtual Duckiedrone Connections

A virtual Duckiedrone runs its services on the base station, but its internal Docker inventory is separate from the base station's practice resources. Choosing the right shell helps you find those services. We will locate the virtual Duckiedrone's container, enter it, and connect each inspection task to the appropriate path.

## Connect to a virtual Duckiedrone

A virtual Duckiedrone runs as a local base-station container. `dts duckiebot virtual connect DUCKIEDRONE_NAME` finds the container named `dts-virtual-DUCKIEDRONE_NAME` and opens an interactive shell with `docker exec`. The connection enters that local container.

Use [Notebook 11](./11-docker-contexts-and-local-targets.ipynb) to identify the base station's local Docker context and check for remote overrides. Replace `LOCAL_DOCKER_CONTEXT` with that context name. The filtered listing shows whether the intended container exists and its current state:

```bash
docker --context LOCAL_DOCKER_CONTEXT container ls --all \
  --filter 'name=^/dts-virtual-DUCKIEDRONE_NAME$'
```

If it is running, open its interactive shell:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

Inside that shell, `docker image ls` and `docker container ls` inspect the virtual Duckiedrone's own Docker daemon and services. This is analogous to running Docker commands in a physical Duckiedrone's SSH shell.

To diagnose a connection error, use `dts duckiebot virtual list` to check its state. For a Duckiedrone listed as `Down`, start it with `dts duckiebot virtual start DUCKIEDRONE_NAME` and reconnect. If it is absent, follow the virtual Duckiedrone setup procedure to create it. The listed state helps you choose between starting an existing Duckiedrone and completing its setup.

## Choose the right connection path

[Table 1](#table-1) connects each goal to the shell or context that reaches its resources.

<table id="table-1" class="lx-table">
  <caption>Table 1: Connection paths for Docker and Duckiedrone goals.</caption>
  <thead>
    <tr>
      <th>Goal</th>
      <th>Connection path</th>
      <th>Docker target</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Complete the local Docker exercises</td>
      <td>Local Docker context</td>
      <td>Learner-owned <code>lx-docker-*</code> resources on the base station</td>
    </tr>
    <tr>
      <td>Inspect a physical Duckiedrone through its own shell</td>
      <td>SSH, then a read-only Docker command</td>
      <td>Docker daemon on the physical Duckiedrone</td>
    </tr>
    <tr>
      <td>Inspect a physical Duckiedrone from the base station</td>
      <td>Explicit Duckiedrone Docker context</td>
      <td>Docker daemon on the physical Duckiedrone</td>
    </tr>
    <tr>
      <td>Open a shell in a running virtual Duckiedrone</td>
      <td><code>dts duckiebot virtual connect DUCKIEDRONE_NAME</code></td>
      <td>Local <code>dts-virtual-DUCKIEDRONE_NAME</code> container</td>
    </tr>
  </tbody>
</table>

The connection path determines which inventory a Docker command inspects. Use the base-station daemon for the `lx-docker-*` exercises, and the Duckiedrone's daemon to observe its services. This keeps practice cleanup separate from the services supporting the virtual or physical Duckiedrone.

## Further reading

See Docker's official guide to [contexts](https://docs.docker.com/engine/manage-resources/contexts/) and the [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) for physical Duckiedrone management.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.

---


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
